<img style="float: center;" src='https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_header.png' alt="stsci_logo" width="900px"/> 

# JWebbinar50: Introduction to Jdaviz 5.0, Part II

_July 7, 2026_

**Authors**: Sarah Betti (STScI), Jackie Champagne (STScI)<br>
**Jdaviz Version**: 5.0.2

- [Documentation](https://jdaviz.readthedocs.io/en/latest/index.html)
- [Source code](https://github.com/spacetelescope/jdaviz)

## Tutorial Outline

This tutorial will demonstrate features recently introduced (since spring 2026) to Jdaviz, with a focus on using combining UI and API calls for working with 3D IFU cubes. 

This notebook will walk you through how to:
- load a 3D data cube using the API
- rearrange viewers and use API hints with copy/paste
- understand data quality ("DQ") flags
- fit and subtract continuum from 1D spectra
- use the Line Lists plugin to determine the redshift
- Bonus: create moment maps

The new features include:
- DQ flags for cubes
- copy/paste from UI to jupyter


In [ ]:
import numpy as np

from astropy.io import fits
from astropy import units as u
from astropy.table import QTable

from specutils import Spectrum, SpectralRegion

from astroquery.mast import Observations

import jdaviz as jd

## Set up our data

Here we load an example 3D IFU cube of the high-redshift AGN host galaxy "DC-417567," observed as part of the ALPINE-CRISTAL-JWST survey with JWST/NIRSpec IFU (PI: Faisst, ID 3045).  The spectrum used here has been published in [Faisst et al., 2026](https://iopscience.iop.org/article/10.3847/1538-4365/ae0928). 

We can load this file directly into jdaviz from the MAST archive using the file name. This is the same as using the UI `url` workflow from Part I. 

The `format` keyword is required and tells jdaviz what format the data has. The options are given in the [documentation](https://jdaviz.readthedocs.io/en/stable/loaders/formats/index.html) for Data Formats. Here, the format should be `3D Spectrum`. 

As a reminder from Part I, if nothing pops open, make sure you allow pop ups on your browser, or change the cell using another options below.

Other options are:
- inline: Will appear below the jd.show(loc='inline') cell
- sidecar: Will appear to the right of the notebook.  This only works in jupyter lab environments. 

In [ ]:
jd.load('mast:JWST/product/jw03045-o057_t057_nirspec_g395m-f290lp_s3d.fits', 
        format='3D Spectrum')

jd.show(loc='popout:window')

We see 3 viewer windows: the <b>top</b> shows the 3D data cube and corresponding DQ array, the <b>middle</b> shows the 3D uncertainty cube, and the <b>bottom</b> shows a 1D spectrum collapsed over the whole cube, which Jdaviz auto-extracted.   

However, we can rearrange the viewers to have a better setup.  By grabbing on the `3D Spectrum (1)` viewer tab, we can drag it up and put it next to the `3D Spectrum` tab, effectively hiding it and giving more space to the data cube and spectrum.   

## Data Quality (DQ) flags

Every JWST dataset run through the JWST calibration pipeline has a data quality mask that flags any pixels that are unreliable or unusable (e.g., dead pixels, hot pixels, etc).  

This array is a bitmask, where the values are integers, formed as combinations of multiple flags for each pixel.  

The most important flag is the `DO_NOT_USE` flag (bit $2^0 = 1$).  For more information on DQ flags with JWST data, see [JDocs](https://jwst-docs.stsci.edu/accessing-jwst-data/jwst-science-data-overview#JWSTScienceDataOverview-Dataqualityarrays(DQ)). 

These flags have been automatically loaded into jdaviz with the cube.  In the `3D Spectrum` viewer, we can see the DQ flag array located in the data menu as `A1`.  <b>Click on the `data menu` to see the full name of the array.</b>

To undertand the what each color means, let's go to the `Data Quality plugin` found in the `Data analysis plugins` menu at the top of the app.

We see there are two main flags: `1 (0)` and `513 (0, 9)`.  Click on the dropdown to see what each stands for.  

## Change Spectral Slice

The current slice in the `3D Spectrum` viewer is not informative.  Let's change the wavelength of the slice using the `Spectral Slice` plugin. 

Each plugin has an API that can be accessed in code cells to interact with that plugin. To access the plugin you can run `jd.plugin['<plugin name'>]`

You can also change it dragging the Slice Tool (the vertical orange bar in the `1D Spectrum` viewer) along the 1D Spectrum. 

In [ ]:
ss = jd.plugins['Spectral Slice']
ss.open_in_tray()
ss.value = 4.38 # in units of microns

## Model Fitting

Our extracted spectrum shows a strong underlying continuum as well as some emission lines.  We can first start by removing this continuum using the `Modeling Fitting` plugin.  

To see all the parameters and methods that are accessible through the API, you can run `dir(mod_fit)`.

In [ ]:
mod_fit = jd.plugins['Model Fitting']

# Running open_in_tray() opens the plugin tray in the UI.
# Note: This is NOT necessary to run API calls, but is useful to see the available features. 
mod_fit.open_in_tray()

dir(mod_fit)

There are lots of tunable parameters.  For this example, we are going to fit a `Spline1D` model to the entire Spectrum.  

To do this,  we first select our dataset and spectral subset. 

In [ ]:
mod_fit.dataset = 'jw03045-o057_t057_nirspec_g395m-f290lp_s3d (sum)'
mod_fit.spectral_subset = 'Entire Spectrum'

We then select and create our model component. 

In [ ]:
mod_fit.model_component = 'Spline1D'
mod_fit.model_component_label = 'S'
mod_fit.create_model_component()

Next, we tell the `Model Fitting` plugin we want to only fit our `Spline1D` component that we have created.  We give it a label and then calculate the fit. 

In [ ]:
mod_fit.equation = 'S'
mod_fit.add_results.label = 'continuum model'
mod_fit.calculate_fit(add_data = True)

The best fit is shown as a blue line.  If you want to adjust this line, go into the `Plot Options` plugin and adjust the color and line thickness! 

# Subtract the model and add it to Jdaviz

Once we have our best fit model, we can subtract it from our data.  

We first extract the dataset. Then, we can load it back into jdaviz. 

In [ ]:
scidata = jd.datasets["jw03045-o057_t057_nirspec_g395m-f290lp_s3d (sum)"].get_data()
moddata = jd.datasets["continuum model"].get_data()
sci_contsub = scidata - moddata

In [ ]:
jd.load(sci_contsub, format='1D Spectrum', data_label='continuum subtracted spectrum')

If we want to turn the visibility of our earlier data off, we can use the `set_layer_visibility` API call from the data menu or click on the eyeball next to the dataset name in the data menu.  To grab the dataset, open up the data menu, and click on the clipboard next to each dataset and copy it into the two `data` and `continuum_model` variables. We then use the `set_layer_visibility` with the command `False` so they are not seen in the viewer.

In [ ]:
data = 'jw03045-o057_t057_nirspec_g395m-f290lp_s3d (sum)'
continuum_model = 'continuum model'

viewers = jd.viewers['1D Spectrum'].data_menu
viewers.set_layer_visibility(data, False)
viewers.set_layer_visibility(continuum_model, False)

Then, press the Home button to reset the zoom so we can see the spectrum! 

## Using Line Lists

We can now see there are some prominant emission lines in our spectrum!  We can try to determine what they are, and their approximate redshift. This isn't a quantitatively robust redshift solution, but it can give us an idea of a rough redshift.

To do this, we go into the `Line Lists` plugin.

In [ ]:
jd.plugins['Line Lists'].open_in_tray()

1. Now, we scroll down to the ------- Present Line Lists ------- section
2. Select the `Available Line Lists` dropdown.
3. We will select the `Galactic 2000A - 11000A` list.
4. Once we have selected it, press the `Load List` button right below.  This will load the list into `jdaviz`.  This list should appear in the ------- Loaded Lines ------ section under a dropdown with a salmon-colored box.
5. Click on the drop down and you will see a selection of lines!  For this exercise, we want to plot the <b>H alpha, H beta, He I 5875.624, [O III] 5006, and [O III] 4958</b> lines.  Using the search bar, you can search for these lines and press the eyeball to plot them (note that you need to include the forbidden bracket if applicable and the space after the atomic species).

None of the lines appeared!  But a arrow with the number 5 appeared in the `1D Spectrum` viewer.  That means our spectrum is very redshifted compared to the rest-frame wavelength of the lines.

## Find the redshift

We now scroll back up in the `Line Lists` plugin to the ------Redshift------ section.  Using the text box, guess the redshift! You can adjust your guess with the draggable bar. When you find it, the lines should line up perfectly with with emission lines!

_Hint: the decimal place is .67 and the bright doublet at 3.3$\mu$m is [O III]._

If you want to determine which line is which on the `1D Spectrum` viewer
1. select the `Enable Line Selection Tools In Spectrum Viewer` box under the ------ Loaded Lines ------ section.
2. Then, click on the vertical lines spectral line identifier tool in the `1D Spectrum` toolbar.
3. Click on a vertical line in the viewer and it will appear in the ------- Identified Line ------- section and tell you what line you have selected!

If we scroll back down to the ------- Loaded Lines ------- section, we will also see the expected observed wavelengths of the emission lines.

## Bonus 1: Fitting Multi-Component Emission Lines

You may have noticed that we said we were dealing with an AGN host galaxy. This galaxy at $z=5.67$ is a Type 1 AGN, meaning that its permitted Balmer lines have both a broad and narrow Gaussian component due to the configuration of the accreting supermassive black hole. We can use the ```Model Fitting``` plugin not only to subtract the continuum, but also to fit the broad + narrow regions of the H alpha line.

First we set up the model fitting plugin like we did before, but this time we will focus on the bright region the spectrum around redshifted H alpha instead of the entire spectrum.

_Reminder: if you want to see this bright line in the IFU viewer, navigate to the ```Spectral Slice``` plugin again and set the slice to 4.375 $\mu$m (or you can specify it in the API like we did above)._

In [ ]:
st = jd.plugins['Subset Tools']
st.import_region(SpectralRegion(4.365 * u.um, 4.395 * u.um), subset_label='Halpha')

mod_fit_agn = jd.plugins['Model Fitting'] 

mod_fit_agn.dataset = 'continuum subtracted spectrum'

mod_fit_agn.spectral_subset = 'Halpha'

Now we can set up the Gaussian fit.

In the ```Model Fitting``` plugin, we can add a ```Gaussian1D``` fit and let Jdaviz calculate the initial parameters. When we press ```Calculate Model```, we should see the Gaussian line appear. <i>Hint: you may need to disable the Spline1D component if you fit the continuum earlier.</i>

What if a broad+narrow model fits this emission line better? and We can add multiple components of the same type to the plugin!

1. Erase your Gaussian1D model (defaulted to the label 'G').
2.  Next, let's add ```Gaussian1D``` twice, labeling one 'broad' and one 'narrow'.
3. Under the ------ Equation Editor ------ section, enter 'narrow + broad'.
4. Press "calculate fit".

If it doesn't look like the fit worked, we can also fix the initial estimates under ------ Model Parameters ------ by setting a value (e.g. for the amplitude, mean, or standard deviation) and checking the box indicating we want to freeze it. Then recalculate the fit and plot. This fit looks visually improved over the single Gaussian model!

In [ ]:
# some API hints - does the same as the UI instructions above

mod_fit_agn.model_component = 'Gaussian1D'
mod_fit_agn.model_component_label = 'narrow'
mod_fit_agn.create_model_component()

mod_fit_agn.model_component = 'Gaussian1D'
mod_fit_agn.model_component_label = 'broad'
mod_fit_agn.create_model_component()


In [ ]:
mod_fit_agn.equation = 'narrow + broad'
mod_fit_agn.fitter = 'LevMarLSQFitter'
mod_fit_agn.add_results.label = 'AGN line'
mod_fit_agn.calculate_fit(add_data = True)

You can combine components for lots of situations. For instance, you could add a Polynomial1D component if you wanted to deal with the non-continuum-subtracted spectrum. Click the dropdown menu under ```Model Component``` to see all the options!


## Bonus 2: Creating a Moment Map

If you want to create a moment map, first create a subset around a spectral emission line and then use this subset in the `Moment Maps` plugin.  

For this example, we select H alpha line.  You can either use the 'Select a range of x values' toolbar button or the `Subset Tools` plugin API calls to create your region around the H alpha lines.  We provide the API calls below. More information on creating and using Subsets will be in Part III.  

In [ ]:
st = jd.plugins['Subset Tools']
st.import_region(SpectralRegion(4.369 * u.um, 4.387 * u.um))

Finally, we open the `Moment Maps` plugin and create the moment map. Here we will do Moment 1, the velocity map, which we will plot in a new viewer.

In [ ]:
mm = jd.plugins['Moment Maps']
mm.open_in_tray()

# we select our spectral subset 
mm.spectral_subset = 'Subset 1'
# the reference wavelength to calculate the moment map
mm.reference_wavelength = 4.377400273
mm.continuum = 'Surrounding'
# we want a moment 1 map, though you can use 0 or 2
mm.n_moment = 1
mm.add_results.label = 'Moment 1'
# this will be created in a new viewer called "Image"
mm.add_results.viewer = 'Image'

mm.calculate_moment()

# use the plot options plugin to change the colormap and vmin/vmax
po = jd.plugins['Plot Options']
po.viewer = 'Image'
po.image_colormap = 'Seismic'
po.stretch_vmin = -500
po.stretch_vmax = 500

---
<img style="float: center;" src='https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_footer.png' alt="stsci_logo" width="400px"/> 